# 第 7 章　事件研究法

::: {.callout-note}
## 本章要点

1. **事件研究的逻辑**：估计期 → 事件窗口 → 正常收益 → 异常收益
2. **基准模型**：市场模型、Fama-French 三因子、Carhart 四因子
3. **统计检验**：$t$ 检验、BMP 检验、Patell 检验
4. **中国市场特殊处理**：停牌、涨跌停对 CAR 的干扰
5. **规范的 CAR 折线图**：含置信带的标准画法

**与前几章的连接**：基准模型就是第 5 章的多元回归（OLS）；
CAR 的截面回归用的是第 6 章的 Fama-MacBeth 逻辑。
合成控制法（单个处理单元的反事实方法）将在后续章节单独介绍。
:::


## 环境准备


In [3]:
# ── 第 7 章　事件研究法　────────────────────────────────────────────────
%pip install --upgrade --force-reinstall "numpy>=1.26,<2.1" "scipy==1.13.1" "statsmodels==0.14.2" "pyfixest>=0.18"

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pyfixest as pf
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_RAW   = 'data_raw'
DATA_CLEAN = 'data_clean'
OUTPUT     = 'output'
for d in [DATA_RAW, DATA_CLEAN, OUTPUT]:
    os.makedirs(d, exist_ok=True)

RNG = np.random.default_rng(42)
print('环境就绪 ✓')


Defaulting to user installation because normal site-packages is not writeable
  Using cached pyfixest-0.50.0-cp312-cp312-win_amd64.whl.metadata (36 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached maketables-0.1.8-py3-none-any.whl.metadata (9.2 kB)
  Using cached narwhals-2.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached interface_meta-1.3.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached great_tables-0.21.0-py3-none-any.whl.metadata (13 kB)
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached commonmark-0.9.1-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached faicons-0.2.2-py3-none

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bqplot 0.12.44 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
deprecated 1.2.18 requires wrapt<2,>=1.10, but you have wrapt 2.1.2 which is incompatible.
inscriptis 2.7.0 requires lxml<6.0.0,>=5.4.0, but you have lxml 6.0.2 which is incompatible.
marker-pdf 1.7.3 requires Pillow<11.0.0,>=10.1.0, but you have pillow 12.1.1 which is incompatible.
openbb-core 1.5.9 requires pandas<3,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
openbb-sec 1.5.1 requires lxml<6.0.0,>=5.2.1, but you have lxml 6.0.2 which is incompatible.
pygam 0.9.1 requires scipy<1.12,>=1.11.1; python_version >= "3.9" and python_version < "3.13", but you have scipy 1.13.1 which is incompatible.
sfma 0.1.0 requires numpy<2.0.0, but you 

ImportError: cannot import name '_lazywhere' from 'scipy._lib._util' (C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\scipy\_lib\_util.py)

---

## 0　引言：市场对事件的第一反应

2021 年 7 月 24 日晚，中国政府发布「双减」政策，限制学科类校外培训机构的商业运营。
次日开盘，教育板块股票集体暴跌——好未来单日跌幅超过 70%，新东方跌超 50%。

这个例子说明了一件事：**股票价格的即时反应，是市场对政策信息的汇总。**
如果市场有效，股价变动的规模和方向，反映了投资者对「这个事件对企业价值影响」的集体判断。

**事件研究法（Event Study）** 正是利用这个逻辑：通过测量事件窗口内的**异常收益率（Abnormal Return）**，量化市场对特定事件的反应，进而推断该事件对企业价值的影响。

本章贯穿案例：**分析「环保政策发布」对重污染行业股票价格的影响**。
我们将从构造模拟事件数据开始，完整走一遍从数据准备到 CAR 图形输出的全流程。


---

## 1　事件研究的基本框架

### 1.1　三个时间窗口

事件研究围绕 **事件日（Event Date，$t=0$）** 展开，把时间轴分为三段：

```
───────────────────────────────────────────────────────────────────▶ t
    [ ----- 估计期 ----- ]  [-- 清洁期 --]  |   [ -- 事件窗口 -- ]
       通常 [-200, -11]       [-10,-2]            [-1, +1]
                                           ↑
                                      事件日 t=0
```

| 窗口 | 作用 | 典型设置 |
|------|------|----------|
| **估计期** | 用正常时期的数据估计基准模型（$\hat{\alpha}$, $\hat{\beta}$）| $[-200, -11]$，约 190 个交易日 |
| **清洁期** | 隔离估计期与事件窗口，避免预期效应污染基准估计 | $[-10, -2]$，约 9 天 |
| **事件窗口** | 计算异常收益率 | 通常 $[-1, +1]$ 或 $[-3, +3]$，视事件性质 |

### 1.2　异常收益率的计算

**第一步：用估计期数据拟合基准模型**

$$R_{it} = \alpha_i + \beta_i R_{mt} + \varepsilon_{it}, \quad t \in \text{估计期}$$

**第二步：用基准模型预测事件窗口内的「正常收益」**

$$E[R_{it} | \Omega_t] = \hat{\alpha}_i + \hat{\beta}_i R_{mt}$$

**第三步：计算异常收益率（AR）**

$$AR_{it} = R_{it} - E[R_{it} | \Omega_t]$$

**第四步：在截面和时间上聚合**

$$AAR_t = \frac{1}{N} \sum_{i=1}^N AR_{it}, \quad
CAR_i[\tau_1, \tau_2] = \sum_{t=\tau_1}^{\tau_2} AR_{it}$$

其中 $AAR_t$ 是第 $t$ 天所有样本的**平均异常收益率**，
$CAR_i[\tau_1, \tau_2]$ 是股票 $i$ 在 $[\tau_1, \tau_2]$ 窗口的**累积异常收益率**。


---

## 2　演示数据准备

我们模拟一个环保政策事件研究场景：
- **50 只重污染行业股票**，每只股票有 260 个交易日的日度收益率
- **事件日 $t=0$**：环保政策发布
- **DGP**：政策在 $[-1, +1]$ 窗口造成平均 $-1.5\%$ 的异常收益（负面冲击）
- 同时提供市场因子和 FF3 因子数据


In [ ]:
# ── 2.1  生成模拟事件研究数据 ────────────────────────────────────────────
N_STOCKS   = 50       # 样本股票数
T_TOTAL    = 260      # 总交易日数
T0         = 210      # 事件日在时间轴上的位置（第 210 天）
EST_START  = -200     # 估计期开始（相对事件日）
EST_END    = -11      # 估计期结束
EVT_START  = -5       # 事件窗口开始
EVT_END    = +5       # 事件窗口结束

# ── 市场因子 ─────────────────────────────────────────────────────────────
market_ret = RNG.normal(0.0003, 0.012, T_TOTAL)   # 市场日度收益率
smb_factor = RNG.normal(0.0001, 0.008, T_TOTAL)   # SMB 因子
hml_factor = RNG.normal(0.0001, 0.007, T_TOTAL)   # HML 因子
mom_factor = RNG.normal(0.0002, 0.009, T_TOTAL)   # 动量因子

# ── 各股票的真实参数 ────────────────────────────────────────────────────
true_alpha = RNG.normal(0.0001, 0.0005, N_STOCKS)
true_beta  = RNG.uniform(0.7, 1.4, N_STOCKS)      # 市场 beta
true_s     = RNG.uniform(-0.3, 0.8, N_STOCKS)     # SMB 暴露
true_h     = RNG.uniform(-0.2, 0.6, N_STOCKS)     # HML 暴露

# ── 生成股票收益率 ───────────────────────────────────────────────────────
# DGP：事件窗口 [-1,+1] 内有 -1.5% 的平均异常收益，股票间有差异
true_car   = RNG.normal(-0.015, 0.025, N_STOCKS)  # 每只股票的真实 CAR

stock_rets = np.zeros((N_STOCKS, T_TOTAL))
for i in range(N_STOCKS):
    eps = RNG.normal(0, 0.015, T_TOTAL)            # 特异性扰动
    r   = (true_alpha[i]
           + true_beta[i]  * market_ret
           + true_s[i]     * smb_factor
           + true_h[i]     * hml_factor
           + eps)
    # 在事件窗口 [-1, +1] 植入异常收益
    for day in [-1, 0, 1]:
        r[T0 + day] += true_car[i] / 3   # 均匀分布在 3 天里
    stock_rets[i] = r

# ── 整理为 DataFrame（长表，每行 = 股票 × 日期）────────────────────────
t_rel = np.arange(T_TOTAL) - T0   # 相对事件日的时间
rows  = []
for i in range(N_STOCKS):
    for t in range(T_TOTAL):
        rows.append({
            'stock_id'  : f'stock_{i:03d}',
            't_abs'     : t,
            't_rel'     : t_rel[t],
            'ret'       : stock_rets[i, t],
            'mkt'       : market_ret[t],
            'smb'       : smb_factor[t],
            'hml'       : hml_factor[t],
            'mom'       : mom_factor[t],
        })

df_evt = pd.DataFrame(rows)

print(f'事件研究数据：{df_evt.shape}')
print(f'股票数：{N_STOCKS}，总交易日：{T_TOTAL}，事件日：t_rel=0')
print(f'真实 CAR[-1,+1] 均值：{true_car.mean()*100:.3f}%（植入值：-1.5%）')
df_evt.head()


---

## 3　基准模型：市场模型与多因子模型

### 3.1　三种常用基准

| 模型 | 公式 | 特点 |
|------|------|------|
| **市场模型** | $R_{it} = \alpha_i + \beta_i R_{mt} + \varepsilon_{it}$ | 最简单，应用最广，适合流动性好的股票 |
| **Fama-French 三因子** | 加入 SMB、HML | 控制规模和价值风格 |
| **Carhart 四因子** | 再加入 MOM（动量）| 控制动量效应，学术论文更常用 |

基准模型越复杂，残差方差越小，检验功效越高——
但也需要足够长的估计期数据和足够稳定的因子暴露度。
对于流动性差的小市值股票，复杂模型的参数估计反而不稳定，
市场模型有时是更稳健的选择。

::: {.callout-note}
## 中国市场的因子数据来源

- **CSMAR**：「中国股票市场交易研究」→「Fama-French 因子」数据表
- **AKShare**：`ak.stock_fama_french_factor_daily()` 可获取日度因子
- **刘阳、吕长江（2019）**重构的 A 股五因子（学术研究更多引用这个版本）
:::


In [ ]:
# ── 3.2  对所有股票估计基准模型（市场模型 + Carhart 四因子）─────────────

def estimate_benchmark(df_stock, t_est_start, t_est_end, model='market'):
    '''
    用估计期数据拟合基准模型，返回参数和残差标准差。
    model: 'market' | 'ff3' | 'carhart'
    '''
    est = df_stock[(df_stock['t_rel'] >= t_est_start) &
                   (df_stock['t_rel'] <= t_est_end)].copy()
    if len(est) < 30:
        return None

    if model == 'market':
        formula = 'ret ~ mkt'
    elif model == 'ff3':
        formula = 'ret ~ mkt + smb + hml'
    else:  # carhart
        formula = 'ret ~ mkt + smb + hml + mom'

    m   = smf.ols(formula, data=est).fit()
    return {'params': m.params, 'sigma': m.resid.std(), 'nobs': len(est)}

# ── 批量估计 ─────────────────────────────────────────────────────────────
benchmarks = {}   # stock_id → {'market': {...}, 'carhart': {...}}
for sid, g in df_evt.groupby('stock_id'):
    benchmarks[sid] = {
        model: estimate_benchmark(g, EST_END + EST_START, EST_END, model)
        for model in ['market', 'ff3', 'carhart']
    }

# 检查估计成功率
ok = sum(1 for v in benchmarks.values() if v['market'] is not None)
print(f'估计成功：{ok}/{N_STOCKS} 只股票')

# 展示第一只股票的市场模型参数
sid0 = list(benchmarks.keys())[0]
p0   = benchmarks[sid0]['market']['params']
print(f'\n{sid0} 市场模型参数：α={p0["Intercept"]:.5f}  β={p0["mkt"]:.4f}')
print(f'真实参数：        α={true_alpha[0]:.5f}  β={true_beta[0]:.4f}')


---

## 4　异常收益率的计算与聚合


In [ ]:
# ── 4.1  计算事件窗口内的 AR ─────────────────────────────────────────────

def compute_ar(df_stock, params, sigma, t_evt_start, t_evt_end, model='market'):
    '''计算事件窗口内每日的异常收益率。'''
    evt = df_stock[(df_stock['t_rel'] >= t_evt_start) &
                   (df_stock['t_rel'] <= t_evt_end)].copy()

    # 预测正常收益
    if model == 'market':
        evt['E_ret'] = params['Intercept'] + params['mkt'] * evt['mkt']
    elif model == 'ff3':
        evt['E_ret'] = (params['Intercept']
                        + params['mkt'] * evt['mkt']
                        + params['smb'] * evt['smb']
                        + params['hml'] * evt['hml'])
    else:
        evt['E_ret'] = (params['Intercept']
                        + params['mkt'] * evt['mkt']
                        + params['smb'] * evt['smb']
                        + params['hml'] * evt['hml']
                        + params['mom'] * evt['mom'])

    evt['AR']    = evt['ret'] - evt['E_ret']
    evt['sigma'] = sigma
    return evt[['t_rel', 'AR', 'sigma']]

# ── 批量计算所有股票的 AR ────────────────────────────────────────────────
ar_records = []
for sid, g in df_evt.groupby('stock_id'):
    bm = benchmarks[sid]['carhart']   # 用 Carhart 四因子
    if bm is None:
        continue
    ar = compute_ar(g, bm['params'], bm['sigma'],
                    EVT_START, EVT_END, model='carhart')
    ar['stock_id'] = sid
    ar_records.append(ar)

df_ar = pd.concat(ar_records, ignore_index=True)
print(f'AR 计算完成：{df_ar.shape}  （{N_STOCKS} 只股票 × {EVT_END-EVT_START+1} 天）')


In [ ]:
# ── 4.2  计算 AAR（截面均值）和 CAR（时间序列累积）────────────────────────

# AAR_t = 第 t 天所有股票的 AR 均值
aar = (df_ar.groupby('t_rel')
       .agg(AAR=('AR','mean'), N=('AR','count'), AAR_std=('AR','std'))
       .reset_index())
aar['AAR_se'] = aar['AAR_std'] / np.sqrt(aar['N'])

# CAAR = AAR 从 t=EVT_START 起的累积
aar = aar.sort_values('t_rel')
aar['CAAR'] = aar['AAR'].cumsum()

# 每只股票的 CAR[-1,+1]
car_window = df_ar[df_ar['t_rel'].between(-1, 1)]
car_by_stock = (car_window.groupby('stock_id')
               .agg(CAR=('AR','sum'), sigma=('sigma','mean'))
               .reset_index())

print('CAAR 时序（事件窗口）：')
print(aar[['t_rel','AAR','CAAR','N']].to_string(index=False))
print(f'\nCAR[-1,+1] 截面均值：{car_by_stock["CAR"].mean()*100:.3f}%')
print(f'真实植入值            ：-1.500%')


---

## 5　统计检验

### 5.1　三种检验方法

| 检验 | 原假设 | 特点 |
|------|--------|------|
| **$t$ 检验（简单）** | $\text{CAAR} = 0$ | 假设所有股票 AR 方差相同，适合同质样本 |
| **Patell 检验** | $\text{AAR}_t = 0$ | 用估计期残差标准化每只股票的 AR，允许方差异质 |
| **BMP 检验** | $\text{CAAR} = 0$ | 考虑了 AR 的横截面相关，对事件聚集（event clustering）更稳健 |

**实践建议**：
- 事件日期分散（不同公司的事件日不同）→ $t$ 检验或 Patell 检验均可
- 事件日期集中（所有公司同一天）→ 必须用 BMP，否则截面相关会严重膨胀 $t$ 值


In [ ]:
# ── 5.2  简单 t 检验 ─────────────────────────────────────────────────────

def t_test_aar(aar_df, t_start, t_end):
    '''对 CAAR[t_start, t_end] 做简单 t 检验。'''
    window = aar_df[(aar_df['t_rel'] >= t_start) &
                    (aar_df['t_rel'] <= t_end)]
    caar   = window['AAR'].sum()
    # 简单标准误：假设各天 AAR 独立
    se     = window['AAR_se'].mean() * np.sqrt(len(window))
    t_stat = caar / se if se > 0 else float('nan')
    p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=window['N'].mean()-1))
    return {'CAAR': caar, 'SE': se, 't': t_stat, 'p': p_val}

for window_name, (t1, t2) in [('[-1,+1]',(-1,1)), ('[-3,+3]',(-3,3)),
                               ('[-5,+5]',(-5,5))]:
    r = t_test_aar(aar, t1, t2)
    sig = '***' if r['p']<0.01 else '**' if r['p']<0.05 else '*' if r['p']<0.1 else ''
    print(f"CAAR{window_name:8s}: {r['CAAR']*100:+.4f}%  ",
          f"t={r['t']:6.3f}  p={r['p']:.4f}  {sig}")


In [ ]:
# ── 5.3  Patell 检验 ─────────────────────────────────────────────────────
# 用估计期残差标准差对每只股票的 AR 做标准化

def patell_test(df_ar_input, t_start, t_end, est_len=190):
    '''
    Patell（1976）检验：
    标准化 AR = AR_it / (sigma_i * sqrt(1 + X_it' (X'X)^{-1} X_it))
    此处用简化版：SAR_it = AR_it / sigma_i
    '''
    window = df_ar_input[(df_ar_input['t_rel'] >= t_start) &
                          (df_ar_input['t_rel'] <= t_end)].copy()
    # 简化 Patell：SAR = AR / sigma
    window['SAR'] = window['AR'] / window['sigma']

    # 每只股票的 SCAR（累积标准化 AR）
    scar = window.groupby('stock_id')['SAR'].sum()
    L2   = t_end - t_start + 1  # 事件窗口长度
    # 调整因子（简化）
    scar_adj = scar / np.sqrt(L2 * (est_len - 2) / (est_len - 4))

    # 截面 t 检验
    N     = len(scar_adj)
    mean  = scar_adj.mean()
    se    = scar_adj.std() / np.sqrt(N)
    t_val = mean / se
    p_val = 2 * (1 - stats.t.cdf(abs(t_val), df=N-1))
    return {'N': N, 'SCAR_mean': mean, 't': t_val, 'p': p_val}

r_patell = patell_test(df_ar, -1, 1)
sig = '***' if r_patell['p']<0.01 else '**' if r_patell['p']<0.05 else '*' if r_patell['p']<0.1 else ''
print(f"Patell 检验 CAR[-1,+1]: SCAR_mean={r_patell['SCAR_mean']:.4f}  ",
      f"t={r_patell['t']:.3f}  p={r_patell['p']:.4f}  {sig}")


In [ ]:
# ── 5.4  BMP 检验 ────────────────────────────────────────────────────────
# Boehmer, Musumeci & Poulsen (1991)：
# 考虑了事件日横截面相关（event clustering），更适合同日事件

def bmp_test(df_ar_input, t_start, t_end):
    '''BMP 检验：截面标准化 AR 后再做时序 t 检验。'''
    window = df_ar_input[(df_ar_input['t_rel'] >= t_start) &
                          (df_ar_input['t_rel'] <= t_end)].copy()
    window['SAR'] = window['AR'] / window['sigma']

    # 每天的截面均值和截面标准差
    daily = window.groupby('t_rel').agg(
        SAAR=('SAR','mean'), SAAR_std=('SAR','std'), N=('SAR','count'))

    # CAAR 和 BMP 检验统计量
    T2    = len(daily)
    CSAR  = daily['SAAR'].sum()
    # BMP 的 t 统计量基于每日 SAAR 的时序标准差
    t_val = CSAR / (daily['SAAR_std'].mean() * np.sqrt(T2))
    p_val = 2 * (1 - stats.t.cdf(abs(t_val), df=T2-1))
    return {'T2': T2, 'CSAR': CSAR, 't': t_val, 'p': p_val}

r_bmp = bmp_test(df_ar, -1, 1)
sig = '***' if r_bmp['p']<0.01 else '**' if r_bmp['p']<0.05 else '*' if r_bmp['p']<0.1 else ''
print(f"BMP 检验 CAR[-1,+1]: CSAR={r_bmp['CSAR']:.4f}  ",
      f"t={r_bmp['t']:.3f}  p={r_bmp['p']:.4f}  {sig}")
print()
print('三种检验汇总：')
for name, t_val, p_val in [
    ('简单 t 检验', t_test_aar(aar,-1,1)['t'], t_test_aar(aar,-1,1)['p']),
    ('Patell 检验', r_patell['t'], r_patell['p']),
    ('BMP 检验'  , r_bmp['t'],    r_bmp['p']),
]:
    sig = '***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.1 else ''
    print(f'  {name:<12}: t={t_val:6.3f}  p={p_val:.4f}  {sig}')


### 5.5　Stata 实现

```stata
%%stata
* ── 事件研究法（Stata，手动实现）────────────────────────────────────────
* 也可以使用外部命令 eventstudy2 或 estudy
* ssc install estudy, replace

use "data_raw/event_daily.dta", clear
* 变量：stock_id, t_rel, ret, mkt, smb, hml, mom

* Step 1：估计市场模型
gen in_est = (t_rel >= -200 & t_rel <= -11)
gen in_evt = (t_rel >= -5  & t_rel <= 5)

* 对每只股票分别回归
statsby alpha=_b[_cons] beta=_b[mkt] sigma=e(rmse), ///
    by(stock_id) saving(params, replace): ///
    reg ret mkt if in_est

* Step 2：合并参数，计算 AR
merge m:1 stock_id using params
gen E_ret = alpha + beta * mkt
gen AR    = ret - E_ret if in_evt

* Step 3：计算 AAR 和 CAAR
collapse (mean) AAR=AR (sd) AR_sd=AR (count) N=AR if in_evt, by(t_rel)
gen CAAR = sum(AAR)

* Step 4：t 检验
gen se   = AR_sd / sqrt(N)
gen t    = AAR / se
```


---

## 6　中国市场特殊处理

中国 A 股市场有几个制度特征，会影响事件研究的可靠性，
在设计研究方案时必须事先考虑。

### 6.1　停牌（Trading Suspension）

**问题**：A 股允许公司申请停牌，停牌期间无价格和收益率数据。
如果公司在事件窗口内停牌，其 AR 无法计算——
而停牌往往与公司本身的重大事件相关（并非随机），
直接删除停牌样本会引入选择偏误。

**处理方案（按严重程度）：**

| 情形 | 处理方式 |
|------|----------|
| 估计期内短暂停牌（< 10 天）| 用可用数据估计，记录停牌天数备查 |
| 估计期内长期停牌（> 30 天）| 排除该股票，说明样本筛选标准 |
| 事件窗口内停牌 | 通常排除，在稳健性检验中讨论影响 |

### 6.2　涨跌停（Price Limit）

**问题**：A 股设有 $\pm 10\%$ 的日涨跌停限制
（ST 股为 $\pm 5\%$，科创板、创业板为 $\pm 20\%$）。
当真实信息冲击超过涨跌停幅度时，价格在事件日当天无法完全调整，
信息会「溢出」到后续几个交易日——即所谓的**价格压制（price truncation）**。

**后果**：如果用 $[-1, +1]$ 窗口，可能低估真实 CAR；
应适当延长事件窗口（如 $[-3, +5]$），或单独分析「触及涨跌停」的子样本。

::: {.callout-warning}
## 涨跌停的识别

在 CSMAR 的 `TRD_Dalyr` 表中，可以通过以下方式识别涨跌停：
- `Dretwd`（含权日收益率）≈ ±10% → 当日触及涨跌停
- 或直接使用 `Clsprc`（收盘价）与前日收盘价计算，比较是否达到限制

**建议**：在事件研究的样本描述中，报告触及涨跌停的观测比例。
:::

### 6.3　信息预期与泄露

中国市场监管执法力度历史上相对宽松，
重大事件（如并购、政策出台）在官方公告前可能已有信息泄露。
**表现**：CAR 在事件日 $t=0$ 之前已经开始累积（即「提前反应」）。

**检验方法**：观察 CAAR 折线图在 $t=0$ 前的走势——
如果 $[-5, -2]$ 段已经出现显著单方向漂移，提示存在信息泄露。
可以将事件日设为市场最早反应日（而非公告日），
或报告多个事件窗口的结果作为稳健性检验。


In [ ]:
# ── 6.4  中国市场特殊处理：停牌和涨跌停的模拟演示 ──────────────────────

# 模拟停牌情况（约 5% 的观测在事件窗口内被标记为停牌）
df_ar_cn = df_ar.copy()
n_suspend = int(len(df_ar_cn) * 0.05)
suspend_idx = RNG.choice(df_ar_cn.index, n_suspend, replace=False)
df_ar_cn.loc[suspend_idx, 'AR'] = np.nan
df_ar_cn['suspended'] = df_ar_cn['AR'].isna()

# 模拟涨跌停（AR 超过 ±9.5% 的截断）
price_limit = 0.095
df_ar_cn['hit_limit'] = (df_ar_cn['AR'].abs() >= price_limit)

print('中国市场数据质量诊断：')
print(f'  停牌观测：{df_ar_cn["suspended"].sum()} 个',
      f'（占 {df_ar_cn["suspended"].mean()*100:.1f}%）')
print(f'  触及涨跌停：{df_ar_cn["hit_limit"].sum()} 个',
      f'（占 {df_ar_cn["hit_limit"].mean()*100:.1f}%）')
print()

# 两种处理方案对 CAR[-1,+1] 的影响
car_full = df_ar[df_ar['t_rel'].between(-1,1)].groupby('stock_id')['AR'].sum()
car_clean = (df_ar_cn
             .loc[~df_ar_cn['suspended'] & ~df_ar_cn['hit_limit']]
             [df_ar_cn['t_rel'].between(-1,1)]
             .groupby('stock_id')['AR'].sum())

print('CAR[-1,+1] 均值对比：')
print(f'  原始样本（含停牌/涨跌停）：{car_full.mean()*100:+.4f}%  N={len(car_full)}')
print(f'  剔除停牌和涨跌停后        ：{car_clean.mean()*100:+.4f}%  N={len(car_clean)}')


---

## 7　规范的 CAR 折线图

CAR 折线图是事件研究论文里最重要的图形，
有一套约定俗成的规范格式。

**标准元素：**
- **主线**：CAAR（截面平均的累积异常收益率）折线
- **置信带**：通常是 95% CI（$\text{CAAR} \pm 1.96 \times \text{SE}$）
- **事件日竖线**：在 $t=0$ 处画垂直虚线
- **零值水平线**：在 $y=0$ 处画水平参考线
- **坐标轴标注**：横轴是「相对事件日的交易天数」，纵轴是百分比
- **图注**：说明样本量 $N$、基准模型、事件窗口、显著性检验结果


In [ ]:
# ── 7.1  标准 CAR 折线图 ─────────────────────────────────────────────────

# 重新计算包含置信区间的 CAAR
aar_plot = (df_ar.groupby('t_rel')
            .agg(AAR=('AR','mean'), N=('AR','count'), SD=('AR','std'))
            .reset_index()
            .sort_values('t_rel'))
aar_plot['SE']   = aar_plot['SD'] / np.sqrt(aar_plot['N'])
aar_plot['CAAR'] = aar_plot['AAR'].cumsum()

# 累积标准误（假设各天 AAR 独立，SE 取最大值简化）
# 更严格的做法是用 AAR 方差的累积和
aar_plot['CAAR_SE'] = aar_plot['SE'].expanding().mean() * np.sqrt(
    range(1, len(aar_plot)+1))
aar_plot['CI_lo'] = aar_plot['CAAR'] - 1.96 * aar_plot['CAAR_SE']
aar_plot['CI_hi'] = aar_plot['CAAR'] + 1.96 * aar_plot['CAAR_SE']

# ── 绘图 ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# 置信带
ax.fill_between(aar_plot['t_rel'],
                aar_plot['CI_lo'] * 100,
                aar_plot['CI_hi'] * 100,
                alpha=0.15, color='steelblue', label='95% 置信带')

# CAAR 主线
ax.plot(aar_plot['t_rel'], aar_plot['CAAR'] * 100,
        'steelblue', linewidth=2, label='CAAR（Carhart 四因子）')

# 参考线
ax.axvline(0, color='red', linewidth=1.2, linestyle='--', alpha=0.7,
           label='事件日 (t=0)')
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')

# 事件窗口标注
ax.axvspan(-1, 1, alpha=0.06, color='orange', label='事件窗口 [-1,+1]')

# 标注关键统计量
caar_11 = aar_plot[aar_plot['t_rel'].between(-1,1)]['AAR'].sum()
t_val   = t_test_aar(aar_plot.rename(columns={'AAR':'AAR'}), -1, 1)['t']
ax.annotate(f'CAAR[-1,+1] = {caar_11*100:+.2f}%\n(t = {t_val:.2f}***)',
            xy=(1, caar_11*100),
            xytext=(2.5, (caar_11-0.01)*100),
            arrowprops=dict(arrowstyle='->', color='black', lw=1),
            fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('相对事件日的交易天数', fontsize=11)
ax.set_ylabel('累积平均异常收益率（%）', fontsize=11)
ax.set_title(f'环保政策对重污染行业股票的影响\n'
             f'（N={N_STOCKS} 只股票，Carhart 四因子基准，估计期 {EST_START}~{EST_END}）',
             fontsize=11)
ax.legend(loc='lower left', fontsize=9)
ax.set_xlim(EVT_START - 0.5, EVT_END + 0.5)

plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch07_car_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('CAR 折线图已保存至 output/ch07_car_plot.png')


::: {.callout-tip}
## 提示词模板：规范 CAR 折线图

````
我有一个事件研究的 CAAR 时序 DataFrame，列为：
t_rel（相对事件日天数）、CAAR（累积均值 AR）、CAAR_SE（标准误）、N（样本量）。

请帮我画一张符合学术论文规范的 CAR 折线图，要求：
1. 主线：CAAR（%），线宽 2，颜色 steelblue
2. 置信带：95% CI（CAAR ± 1.96 × CAAR_SE），半透明填充
3. 事件日 t=0 处加红色虚线
4. y=0 处加灰色参考线
5. 在 [-1,+1] 区域加半透明背景色
6. 在图内标注 CAAR[-1,+1] 的值和 t 统计量
7. 图注说明：样本量 N、基准模型名称、事件窗口
8. 保存至 output/car_plot.png（dpi=150）
````
:::


---

## 9　综合案例：完整事件研究流程


In [ ]:
# ── 9.1  一键运行完整流程 ────────────────────────────────────────────────

def run_event_study(df_input, model='carhart',
                    est_start=-200, est_end=-11,
                    evt_start=-5, evt_end=5,
                    car_window=(-1, 1)):
    '''
    完整事件研究流程：
    1. 估计基准模型
    2. 计算 AR
    3. 聚合 AAR / CAAR
    4. 三种检验
    5. 绘制 CAR 图
    '''
    print(f'事件研究法  基准模型={model}  估计期=[{est_start},{est_end}]  '
          f'事件窗口=[{evt_start},{evt_end}]')
    print('─' * 65)

    # Step 1：估计基准模型
    bms = {}
    for sid, g in df_input.groupby('stock_id'):
        bms[sid] = estimate_benchmark(g, est_end + est_start, est_end, model)
    n_ok = sum(1 for v in bms.values() if v is not None)
    print(f'Step 1 完成：{n_ok} 只股票估计成功')

    # Step 2：计算 AR
    ar_list = []
    for sid, g in df_input.groupby('stock_id'):
        if bms[sid] is None: continue
        ar = compute_ar(g, bms[sid]['params'], bms[sid]['sigma'],
                        evt_start, evt_end, model)
        ar['stock_id'] = sid
        ar_list.append(ar)
    df_ar_out = pd.concat(ar_list, ignore_index=True)
    print(f'Step 2 完成：AR 计算，共 {len(df_ar_out)} 个观测')

    # Step 3：AAR / CAAR
    aar_out = (df_ar_out.groupby('t_rel')
               .agg(AAR=('AR','mean'), N=('AR','count'), SD=('AR','std'))
               .reset_index().sort_values('t_rel'))
    aar_out['SE']   = aar_out['SD'] / np.sqrt(aar_out['N'])
    aar_out['CAAR'] = aar_out['AAR'].cumsum()
    print(f'Step 3 完成：CAAR 时序计算')

    # Step 4：检验
    print(f'Step 4 检验结果（CAR{car_window}）：')
    r_t   = t_test_aar(aar_out, *car_window)
    r_p   = patell_test(df_ar_out, *car_window)
    r_b   = bmp_test(df_ar_out, *car_window)
    for name, r in [('t 检验', r_t), ('Patell', r_p), ('BMP', r_b)]:
        t_v = r.get('t', float('nan'))
        p_v = r.get('p', float('nan'))
        sig = '***' if p_v<0.01 else '**' if p_v<0.05 else '*' if p_v<0.1 else ''
        print(f'  {name:<8}: t={t_v:6.3f}  p={p_v:.4f} {sig}')

    return df_ar_out, aar_out

df_ar_final, aar_final = run_event_study(df_evt, model='carhart')


---

## 10　章末练习

**练习 1（基准模型比较）**
分别用市场模型、FF3、Carhart 四因子三种基准模型计算 CAAR[-1,+1]，
比较三种模型的估计结果（系数均值、标准误、$t$ 值）。
哪种模型的残差标准差最小？这对检验功效有什么影响？

**练习 2（中国市场稳健性）**
在本章的模拟数据中，将 20% 的事件窗口观测随机设为「停牌」（缺失），
将 10% 的观测的 AR 截断至 ±9.5%（模拟涨跌停）。
比较处理前后 CAAR[-1,+1] 的大小和显著性，
讨论这两种问题对事件研究结论的影响方向。

**练习 3（CAR 截面回归）**
把 `CAR[-1,+1]`（每只股票）作为被解释变量，
用公司特征（规模、杠杆、国企哑变量等）做截面回归，
检验「哪类公司受事件冲击更大」。
这是第 6 章 Fama-MacBeth 逻辑的单期截面应用。

**练习 4（思考题）**
有研究者发现：某政策公告前 3 天，样本公司的 CAAR 已经开始显著下滑。
这有哪几种可能的解释？
如何在研究设计上区分「信息泄露」和「事件日认定错误」这两种解释？
